# PhoBERT Spam Classification for E-Commerce Reviews

Đây là sổ tay (Notebook) được tự động tạo để huấn luyện mô hình PhoBERT phân loại đánh giá rác (Spam) trên Kaggle.

**Lưu ý trước khi chạy:**
1. Đảm bảo bạn đã bật **Accelerator** là `GPU T4 x2` ở cột bên phải màn hình.
2. Đảm bảo bạn đã Upload file `spam_dataset.csv` vào Input.

In [ ]:
!pip install transformers accelerate scikit-learn pandas

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import os

# 1. Khởi tạo Dataset Class
class SpamDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True,
            return_attention_mask=True, return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# 2. Đọc Dữ liệu (Sửa đường dẫn nếu tên thư mục Dataset của bạn khác)
try:
    # Thường khi upload lên Kaggle, file sẽ nằm ở /kaggle/input/[tên-thư-mục]/gemma_labeled_spam.csv
    csv_path = "/kaggle/input/gemma-spam-dataset/gemma_labeled_spam.csv"
    df = pd.read_csv(csv_path)
    # Lọc bỏ các nhãn lỗi (-1) để tránh lỗi CUDA device-side assert
    df = df[df['gemma_is_spam'].isin([0, 1])]
except FileNotFoundError:
    print("❌ Lỗi: Không tìm thấy file data. Bạn hãy kiểm tra lại đường dẫn trong thư mục /kaggle/input/")
    # Nếu chạy ở máy tính local:
    # df = pd.read_csv("data/gemma_labeled_spam.csv")

print(f"Tổng số bình luận: {len(df)}")
print(df['gemma_is_spam'].value_counts())

In [ ]:
# 3. Tải Model PhoBERT và Tokenizer
X_train, X_test, y_train, y_test = train_test_split(df['comment'], df['gemma_is_spam'], test_size=0.1, random_state=42)

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=2)

train_dataset = SpamDataset(X_train.tolist(), y_train.tolist(), tokenizer)
test_dataset = SpamDataset(X_test.tolist(), y_test.tolist(), tokenizer)

In [ ]:
# 4. Cấu hình Tham số Huấn luyện
training_args = TrainingArguments(
    output_dir='./results_spam',
    num_train_epochs=3,
    per_device_train_batch_size=16, # GPU T4 có thể gánh batch 16
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_spam',
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# 5. Huấn Luyện (Kaggle T4 x2 mất khoảng 5-10 phút cho 26,000 dòng)
print("Bắt đầu huấn luyện...")
trainer.train()

In [ ]:
# 6. Lưu Model hoàn thiện
model.save_pretrained("/kaggle/working/phobert-spam-filter")
tokenizer.save_pretrained("/kaggle/working/phobert-spam-filter")
print("Hoàn tất! Hãy nén thư mục phobert-spam-filter (ở thanh bên phải) và tải về máy.")